In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 59.8 MB/s eta 0:00:00


In [2]:
import faiss
import numpy as np
import pandas as pd
import pickle
import json
import time
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
import re
import scipy.sparse as sp
import warnings
warnings.filterwarnings("ignore")

df = pd.read_parquet("/kaggle/input/notebooks/anymansince2005/baseline-model/arxiv_ml_with_corpus.parquet")
df = df.reset_index(drop = True)

with open("/kaggle/input/notebooks/anymansince2005/baseline-model/baseline_metrics.json") as f:
    baseline_metrics = json.load(f)

print(f"Loaded {len(df):,} papers")
print(f"Baseline precision@10 to beat : {baseline_metrics['mean_precision@k']}")

Loaded 110 papers
Baseline precision@10 to beat : 0.48


In [3]:
def build_embeddings_input(df:pd.DataFrame) -> list[str]:
    texts = []
    for _,row in df.iterrows():
        text = f"Title: {str(row['title']).strip()}"
        text += " "
        text += f"Abstract: {str(row['abstract']).strip()}"
        texts.append(text)
    return texts

embedding_inputs = build_embeddings_input(df)
print(f"\nSample embedding input:")
print(embedding_inputs[0][:300])


Sample embedding input:
Title: Intelligent location of simultaneously active acoustic emission sources:
  Part I Abstract: The intelligent acoustic emission locator is described in Part I, while Part
II discusses blind source separation, time delay estimation and location of two
simultaneously active continuous acoustic em


In [4]:
print("\nLoading sentence-transformer model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Model max sequence length: {model.max_seq_length}")
print(f"Embedding dimensions     : {model.get_sentence_embedding_dimension()}")

print("\nGenerating embeddings ")
start = time.time()

embeddings = model.encode(
    embedding_inputs,
    batch_size = 128,
    show_progress_bar = True,
    convert_to_numpy = True,
    normalize_embeddings = True,
)

elapsed = time.time() - start
print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Time taken        : {elapsed:.1f}s")
print(f"Dtype             : {embeddings.dtype}")


Loading sentence-transformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model max sequence length: 256
Embedding dimensions     : 384

Generating embeddings 


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embeddings shape: (110, 384)
Time taken        : 8.6s
Dtype             : float32


In [5]:
print("\nBuilding FAISS index...")

EMBEDDING_DIM = embeddings.shape[1]

def build_faiss_index(embeddings:np.ndarray) -> faiss.Index:
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    
    index_with_ids = faiss.IndexIDMap(index)
    ids = np.arange(len(embeddings)).astype(np.int64)
    
    index_with_ids.add_with_ids(
        embeddings.astype(np.float32),
        ids,
    )
    return index_with_ids

faiss_index = build_faiss_index(embeddings)
print(f"FAISS index total vectors : {faiss_index.ntotal:,}")


Building FAISS index...
FAISS index total vectors : 110


In [6]:
def search_by_idx(
    idx : int,
    faiss_index,
    embeddings:np.ndarray,
    df : pd.DataFrame,
    top_n : int=10,) -> pd.DataFrame:
    
    query_vec = embeddings[idx].reshape(1,-1).astype(np.float32)
    scores, indices = faiss_index.search(query_vec, top_n+1)
    
    results = [
        (int(i), float(s))
        for i,s in zip(indices[0], scores[0])
        if i!= idx
    ][:top_n]
    
    rec_df = df.iloc[[r[0] for r in results]][
      ["paper_id","title","categories","date"]
    ].copy()
    rec_df["similarity_score"] = [round(r[1], 4) for r in results]
    return rec_df.reset_index(drop=True)

def search_by_text(
    query : str,
    model,
    faiss_index,
    df: pd.DataFrame,
    top_n : int=10,) -> pd.DataFrame:

    query_vec = model.encode(
        [query],
        normalize_embeddings = True,
        convert_to_numpy = True,
    ).astype(np.float32)
    
    scores,indices = faiss_index.search(query_vec,top_n)

    rec_df = df.iloc[indices[0]][
       ["paper_id","title","categories","date"]
    ].copy()
    rec_df["similarity_score"] = [round(float(s), 4) for s in scores[0]]
    return rec_df.reset_index(drop=True)

def search_by_multiple_papers(
    indices_list: list[int],
    embeddings : np.ndarray,
    model,
    faiss_index,
    df: pd.DataFrame,
    top_n : int=10,) -> pd.DataFrame:

    profile_vec = embeddings[indices_list].mean(axis=0,keepdims=True)
    profile_vec = profile_vec/np.linalg.norm(profile_vec)
    profile_vec = profile_vec.astype(np.float32)
    
    scores,found_indices = faiss_index.search(profile_vec,top_n+len(indices_list))

    results = [
        (int(i), float(s))
        for i,s in zip(found_indices[0], scores[0])
        if i not in indices_list
    ][:top_n]
    
    rec_df = df.iloc[[r[0] for r in results]][
       ["paper_id","title","categories","date"]
    ].copy()
    rec_df["similarity_score"] = [round(r[1], 4) for r in results]
    return rec_df.reset_index(drop=True)

In [7]:
print("\n"+"="*60)
print("TEST 1 - Using existing paper index")
print("="*60)

sample_idx = 42
print(f"\nQuery: {df.iloc[sample_idx]['title']}")
recs1 = search_by_idx(sample_idx, faiss_index, embeddings, df, top_n=10)
print(recs1[["title","similarity_score","categories"]].to_string())

print("\n"+"="*60)
print("TEST 2 - Using free-text query")
print("="*60)

query = "contrastive learning self-supervised visual representations"
print("\nQuery: '{Query}'")
recs2 = search_by_text(query, model, faiss_index, df, top_n=10)
print(recs2[["title","similarity_score","categories"]].to_string())

print("\n"+"="*60)
print("TEST 3 - Using multiple papers")
print("="*60)

read_indices = [0,5,18,42]
print(f"\nRead papers:")
for i in read_indices:
    print(f"[{i}]{df.iloc[i]['title']}")

recs3 = search_by_multiple_papers(read_indices, embeddings, model, faiss_index, df, top_n=10)
print("\nRecommendations based on reading profile:")
print(recs3[["title","similarity_score","categories"]].to_string())


TEST 1 - Using existing paper index

Query: The Parameter-Less Self-Organizing Map algorithm
                                                                                                                       title  similarity_score                categories
0                                                      Traitement Des Donnees Manquantes Au Moyen De L'Algorithme De Kohonen            0.4748          [stat.AP, cs.NE]
1                             Fuzzy Artmap and Neural Network Approach to Online Processing of Inputs\n  with Missing Values            0.3354                   [cs.AI]
2  Exploiting Heavy Tails in Training Times of Multilayer Perceptrons: A\n  Case Study with the UCI Thyroid Disease Database            0.3331                   [cs.NE]
3                                                                            A neural network approach to ordinal regression            0.3298     [cs.LG, cs.AI, cs.NE]
4                                               Statistical M

In [8]:
def precision_at_k(recom_cats, query_cats, k=10):
    hits = sum(
        1 for cats in recom_cats[:k]
        if set(cats) & set(query_cats)
    )
    return hits/k

def eval_embedding_model(df, faiss_index, embeddings, sample_size=110, k=10):
    indices = np.random.choice(len(df), size = 110, replace=False)
    precisions = []

    for idx in indices:
        query_cats = df.iloc[idx]["categories"]
        recs = search_by_idx(idx, faiss_index, embeddings, df, top_n=k)
        p_at_k = precision_at_k(recs["categories"].tolist(), query_cats, k)
        precisions.append(p_at_k)
    
    return {
        "mean_precision@k"   : round(np.mean(precisions), 4),
        "median_precision@k" : round(np.median(precisions), 4),
        "std"              : round(np.std(precisions), 4),
        "min"              : round(np.min(precisions), 4),
        "max"              : round(np.max(precisions), 4),
        "k"                : k,
        "sample_size"      : sample_size,
    }

print("\n"+"="*60)
print("Model Comparision")
print("="*60)

np.random.seed(42)
embedding_metrics = eval_embedding_model(df, faiss_index, embeddings, sample_size=200, k=10)

print(f"\n{'Metric':<25}{'TF-IDF BASELINE':>18}{'Sentence-BERT':>18}{'Delta':>10}")
print("-"*75)
for key in ["mean_precision@k", "median_precision@k","std","min","max"]:
    base = baseline_metrics.get(key,0)
    new = embedding_metrics.get(key,0)
    delta = round(new-base,4)
    arrow = "▲" if delta>0 else "▼"
    print(f"{key:<23}{base:>18}{new:>18} {arrow}{abs(delta):>8}")


Model Comparision

Metric                      TF-IDF BASELINE     Sentence-BERT     Delta
---------------------------------------------------------------------------
mean_precision@k                     0.48            0.5491 ▲  0.0691
median_precision@k                    0.5               0.6 ▲     0.1
std                                0.2467            0.2471 ▲  0.0004
min                                   0.0               0.0 ▼     0.0
max                                   1.0               1.0 ▼     0.0


In [9]:
print("\n"+"="*60)
print("Qualitative Comparision")
print("="*60)

tfidf_matrix = sp.load_npz("/kaggle/input/notebooks/anymansince2005/baseline-model/tfidf_matrix.npz")
with open("/kaggle/input/notebooks/anymansince2005/baseline-model/tfidf_vectorizer.pkl","rb") as f:
    vectorizer = pickle.load(f)

def clean_text(text):
    if not isinstance(text,str):return ""
    text = text.lower()
    text = re.sub(r"\$\$.*?\$\$"," ",text, flags=re.DOTALL)
    text = re.sub(r"\$.*?\$", " ",text)
    text = re.sub(r"[^a-z0-9\s]"," ",text)
    return re.sub(r"\s+", " ",text).strip()

Query =  "federated learning privacy preserving distributed machine learning"
print(f"\nQuery: '{Query}'")

tfidf_vec = vectorizer.transform([clean_text(Query)])
tfidf_scores = linear_kernel(tfidf_vec, tfidf_matrix).flatten()
tfidf_top = tfidf_scores.argsort()[::-1][:5]

print("\nTF-IDF Top 5:")
for rank, idx in enumerate(tfidf_top,1):
    print(f"{rank}.[{tfidf_scores[idx]:.4f}]{df.iloc[idx]['title']}")

emb_recs = search_by_text(Query, model, faiss_index, df, top_n=5)
print("\nSentence-BERT Top 5:")
for rank, row in emb_recs.iterrows():
    print(f"{rank+1}.[{row['similarity_score']:.4f}]{row['title']}")


Qualitative Comparision

Query: 'federated learning privacy preserving distributed machine learning'

TF-IDF Top 5:
1.[0.2052]An Adaptive Strategy for the Classification of G-Protein Coupled
  Receptors
2.[0.1426]Ensemble Learning for Free with Evolutionary Algorithms ?
3.[0.0978]Parametric Learning and Monte Carlo Optimization
4.[0.0801]Design, Implementation, and Cooperative Coevolution of an Autonomous/
  Teleoperated Control System for a Serpentine Robotic Manipulator
5.[0.0790]Compressed Regression

Sentence-BERT Top 5:
1.[0.3331]Compressed Regression
2.[0.3242]Learning from compressed observations
3.[0.2389]Lasso type classifiers with a reject option
4.[0.2369]Stochastic Optimization Algorithms
5.[0.2341]On the monotonization of the training set


In [10]:
print("\n"+"="*60)
print("Saving model artifacts")
print("="*60)

np.save("/kaggle/working/embeddings.npy", embeddings)
print("Saved: embeddings.npy")

faiss.write_index(faiss_index, "/kaggle/working/faiss_index.bin")
print("Saved: faiss_index.bin")

with open("/kaggle/working/embedding_metrics.json","w") as f:
    json.dump(embedding_metrics, f, indent=2)
print("Saved: embedding_metrics.json")

all_metrics = {
    "tfidf_baseline" : baseline_metrics,
    "sentence_bert"  : embedding_metrics,
}
with open("/kaggle/working/all_metrics.json","w") as f:
    json.dump(all_metrics, f, indent=2)
print("Saved: all_metrics.json")


Saving model artifacts
Saved: embeddings.npy
Saved: faiss_index.bin
Saved: embedding_metrics.json
Saved: all_metrics.json
